In [ ]:
import os
import zipfile
import tempfile
import shutil

from tabulate import tabulate
import SimpleITK as sitk

import csv
import pydicom
from pydicom.tag import Tag

from collections import defaultdict
import numpy as np
import sys

import matplotlib.pyplot as plt
import re
from IPython.display import clear_output


In [ ]:
from script import keep_log, unzip_recursively, get_stl_files
from script import is_dicom_file, append_middle_dicom_tags_to_csv

from script import organize_dicom_series, get_shape, anonymize_dicom_series

from script import get_nifti_images_and_mask_from_managed_patient_dir, process_dicom_folder, get_mask_from_stl

In [ ]:
input_path = "./pipeline/plai data 2"
save_path = "./pipeline/unzipped_dir"

base_name = os.path.splitext(os.path.basename(input_path))[0] if os.path.isfile(input_path) else os.path.basename(input_path)
unzipped_directory_path = os.path.join(save_path, base_name)
os.makedirs(unzipped_directory_path, exist_ok=True)

unzip_recursively(input_path, unzipped_directory_path)

In [ ]:
stl_files = get_stl_files(unzipped_directory_path)
if not stl_files:
    print("No stl files found")
    
for files in stl_files:
    print(os.path.basename(files))

In [ ]:
walked_series_path = []
for dirpath, dirnames, filenames in os.walk(unzipped_directory_path):
    # Skip if no files in the directory
    if filenames:
        for files in filenames:
            file_path = os.path.join(dirpath, files)
            if is_dicom_file(file_path):
                if dirpath not in walked_series_path:
                    walked_series_path.append(dirpath)
                    append_middle_dicom_tags_to_csv(f"{unzipped_directory_path}/metadata.csv", dirpath)
                    print(dirpath)

In [ ]:
save_path = "./pipeline/organized_dir"
organized_save_directory = os.path.join(save_path, os.path.basename(unzipped_directory_path))
if os.path.exists(organized_save_directory):
    shutil.rmtree(organized_save_directory)

organize_dicom_series(unzipped_directory_path, organized_save_directory)
stl_files = []
stl_files = get_stl_files(unzipped_directory_path)
if stl_files:
    for paths in stl_files:
        os.makedirs(os.path.join(organized_save_directory, "stl_files"), exist_ok=True)
        shutil.copy(paths, os.path.join(organized_save_directory, "stl_files"))
    

In [ ]:
rows = []
for dirpath, dirnames, filenames in os.walk(organized_save_directory):
    # Skip if no files in the directory
    if filenames:
        # Pick the first file (can add filter for .dcm if needed)
        first_file = filenames[0]
        file_path = os.path.join(dirpath, first_file)
        series_path = dirpath
        series_name = os.path.basename(series_path)
        # print("file_path:", file_path)
        # print("series_path:", series_path)
        file_size, file_count = get_shape(series_path)
        if file_count >1:
            rows.append([series_name, file_size, file_count])
        # print(file_size)
        # print("-" * 80)
        dirnames[:] = []  # Clear the list of dirnames to stop recursion at this level

print(tabulate(rows, headers=["Series Name", "Dimension", "Total Files"], tablefmt="pretty"))

In [ ]:
for dirpath, dirnames, filenames in os.walk(organized_save_directory):
    # Skip if no files in the directory
    if filenames:
        series_path = dirpath
        anonymize_dicom_series(series_path)
        dirnames[:] = []

In [ ]:
walked_series_path = []
for dirpath, dirnames, filenames in os.walk(organized_save_directory):
    # Skip if no files in the directory
    if filenames:
        series_path = dirpath
        first_file = filenames[0]
        file_path = os.path.join(series_path, first_file)
        if is_dicom_file(file_path):
            if series_path not in walked_series_path:
                walked_series_path.append(series_path)
                append_middle_dicom_tags_to_csv(f"{organized_save_directory}/anonymized_metadata.csv", series_path)
                print(series_path)

In [ ]:
processed_save_path = os.path.join("./pipeline/Processed_nifti_images", os.path.basename(organized_save_directory))
stl_files = get_stl_files(organized_save_directory)

for dirpath, dirnames, filenames in os.walk(organized_save_directory):
    if filenames:
        series_path = dirpath
        for files in os.listdir(series_path):
            file_path = os.path.join(series_path, files)
            if is_dicom_file(file_path):
                if len(os.listdir(series_path)) <20:
                    break
                save_path = os.path.join(processed_save_path, os.path.basename(os.path.dirname(os.path.dirname(series_path))), os.path.basename(os.path.dirname(series_path)), os.path.basename(series_path))
                os.makedirs(save_path, exist_ok=True)
                try:
                    process_dicom_folder(series_path, save_path)
                    if len(stl_files)!= 0:
                        get_mask_from_stl(series_path, save_path, stl_files)
                except Exception as e:
                    print(f"Failed processing {series_path}: \n {e}")
            break

        # dirnames[:] = []

In [ ]:
# def print_information(image_path):
#     image = sitk.ReadImage(im_path)

#     print("Spacing:", image.GetSpacing())
#     print("Origin:", image.GetOrigin())
#     print("Direction:", image.GetDirection())

#     # Convert to numpy array to get shape
#     array = sitk.GetArrayFromImage(image)
#     print("Shape (z, y, x):", array.shape)

# im_path = "./pipeline/Processed_nifti_images/plai data 2/ST000001/SE000004/ANKLE_3/ANKLE_3_image.nii.gz"
# print_information(im_path)
# im_path = "./pipeline/Processed_nifti_images/plai data 2/ST000001/SE000004/ANKLE_3/cheville.nii.gz"
# print_information(im_path)


In [ ]:
from script import visualize_nifti_middle_slices_sitk, extract_axial_slice_range, show_three_coronal_slices, extract_axial_slice_range_in_mask
from script import compare_mask_alignment, manage_hierarchy_for_mask

First of all, it visualizes the nifti image within cell

it asks user for input for start and end slice index to take specific slices in image | 
example input: 0, 245

again it asks for region name | 
example input: ankle

if input is given like ankle, re it starts over again (In case server runs slow and fails to visualize)

In [ ]:

save_path = "./pipeline/final_processing"
final_save_path = os.path.join(save_path, os.path.basename(processed_save_path))

for dirpath, dirnames, filenames in os.walk(processed_save_path):
    if filenames:
        series_path = dirpath
        for files in os.listdir(series_path):
            if "image.nii.gz" in files:
                nifti_image_path = os.path.join(series_path, files)
                series_save_path = os.path.join(final_save_path, os.path.basename(os.path.dirname(os.path.dirname(series_path))), os.path.basename(os.path.dirname(series_path)))
                os.makedirs(series_save_path, exist_ok = True)
                repeat = "re"
                while repeat == "re":
                    visualize_nifti_middle_slices_sitk(nifti_image_path)
                    re_input = True
                    while re_input:
                        input_str = input("Enter start and end slice indices from the approximation for the visualization given below: ")
                        start_slice, end_slice = map(int, input_str.split(','))
                        
                        img = sitk.ReadImage(nifti_image_path)
                        num_slices = img.GetSize()[2]  # (x, y, z)
                        if start_slice <0 or end_slice > num_slices:
                            re_input = True
                        else:
                            re_input = False

                    user_input = input(f"Enter anatomy (pelvis, knee, ankle) you are processing from the visualization given below: \n (Hint: {os.path.basename(nifti_image_path)}")
                    if ',' in user_input:
                        region, repeat = user_input.split(',', 1)
                        region = region.strip()
                        repeat = repeat.strip()
                    else:
                        region = user_input.strip()
                        repeat = "no"
                    images = extract_axial_slice_range(nifti_image_path, series_save_path, region, start_slice, end_slice-1)
                    clear_output(wait=True)
                    print("Checking mask...")
                    for file in os.listdir(series_path):
                        if not "image.nii.gz" in file:
                            mask_path = os.path.join(series_path, file)
                            show_three_coronal_slices(nifti_image_path, mask_path)
                            input_region = input(f"Enter anatomy (pelvis, knee, ankle, skip) you are processing from the visualization given below: \n (Hint: {os.path.basename(mask_path)} ")
                            if input_region == "skip":
                                continue
                            extracted_mask = extract_axial_slice_range_in_mask(mask_path, series_save_path, region, start_slice, end_slice-1)
                            if images == "unilateral":
                                output_save_dir = os.path.join(output_dir, f"{images}_region")
                                output_path = os.path.join(output_save_dir, file)
                                sitk.WriteImage(extracted_mask, output_path)
                            if images == "bilateral":
                                better = compare_mask_alignment(extracted_mask, os.path.join(series_save_path, f"left_{region}", f"{region}_image.nii.gz"), os.path.join(series_save_path, f"right_{region}", f"{region}_image.nii.gz"), tol=1e-3)
                                manage_hierarchy_for_mask(series_save_path, extracted_mask, region, input_region, better)

                                    


